In [ ]:
%load_ext autoreload
%autoreload 2
import os
import sys

sys.path.append("../")

In [ ]:
import numpy as np
import pandas as pd
from tqdm import tqdm

import src.data_preprocessing.caption_generation as cg
import src.data_preprocessing.create_aux_data as cad
import src.data_preprocessing.data_utils as du
import src.data_preprocessing.gee_utils as gu
from src.data.base_caption_builder import BaseCaptionBuilder, DummyCaptionBuilder
from src.data.base_datamodule import BaseDataModule
from src.data.butterfly_caption_builder import ButterflyCaptionBuilder
from src.data.butterfly_dataset import ButterflyDataset

## Images

In [ ]:
datadir = "/Users/tplas/data/aether_data/s2bms/source/alphaearth_av-128/"
tiffiles = os.listdir(datadir)
tiffiles = [f for f in tiffiles if f.endswith(".tif")]
list_names = []
for i_f, f in tqdm(enumerate(tiffiles)):
    fp = os.path.join(datadir, f)
    im = du.load_tiff(fp)
    assert (
        im.ndim == 3 and im.shape[1] == 1 and im.shape[2] == 1
    ), f"Expected image to have shape (bands, 1, 1), but got {im.shape}"
    if i_f == 0:
        vals = np.squeeze(im)[:, None]
    else:
        vals = np.concatenate((vals, np.squeeze(im)[:, None]), axis=1)
    list_names.append("_".join(f.split("_")[1:4]))


n_embed = 64
assert (
    vals.shape[0] == n_embed
), f"Expected number of bands to be {n_embed}, but got {vals.shape[0]}"

In [ ]:
df_vals = pd.DataFrame(vals.T, columns=["emb_" + str(i) for i in range(n_embed)])
df_vals["name_loc"] = list_names
df_vals["ind_int"] = [int(x.split("_")[1]) for x in list_names]
df_vals = df_vals.sort_values("ind_int").reset_index(drop=True)
df_vals = df_vals.drop(columns=["ind_int"])
df_vals.to_csv(os.path.join(datadir, "aef-uk-unlabelled_average-128.csv"), index=False)

In [ ]:
df_vals

## CSV data

In [ ]:
folder = "/Users/tplas/data/aether_data/satbird-USA-summer/source/dynamicworld/"
assert os.path.exists(folder), f"Folder {folder} does not exist"

In [ ]:
for i_f, f in enumerate(os.listdir(folder)):
    fp = os.path.join(folder, f)
    tmp_df = pd.read_csv(fp)
    if i_f == 0:
        df = tmp_df
    else:
        df = pd.concat((df, tmp_df), axis=0)

df_lc = df.copy()
df_lc = df_lc.rename(columns={"name": "name_loc"})
df_lc = df_lc.rename(
    columns={
        c: f"aux_{c.replace('dynamicworld_', 'dw_')}"
        for c in df_lc.columns
        if c.startswith("dynamicworld")
    }
)

In [ ]:
## Get top 3 land cover classes for each data point
LC_NAMES = df_lc.columns[:9]

df_lc["aux_dw_top_1"] = df_lc[LC_NAMES].idxmax(axis=1).str.replace("aux_", "")
df_lc["aux_dw_top_2"] = (
    df_lc[LC_NAMES].apply(lambda x: x.nlargest(2).index[1], axis=1).str.replace("aux_", "")
)
df_lc["aux_dw_top_3"] = (
    df_lc[LC_NAMES].apply(lambda x: x.nlargest(3).index[2], axis=1).str.replace("aux_", "")
)

In [ ]:
df_lc

In [ ]:
df_model_ready = pd.read_csv(
    "/Users/tplas/data/aether_data/satbird-USA-summer/model_ready_satbird-USA-summer.csv"
)
df_merged = df_model_ready.merge(df_lc, on="name_loc", how="left")
df_merged.to_csv(
    "/Users/tplas/data/aether_data/satbird-USA-summer/model_ready_satbird-USA-summer_with_lc.csv",
    index=False,
)

In [ ]:
df_merged

In [ ]:
df_merged.columns[:-20]

In [ ]:
path_s2bms = "/Users/tplas/data/aether_data/s2bms/model_ready_s2bms.csv"
assert os.path.exists(path_s2bms), f"File {path_s2bms} does not exist"

df_s2bms = pd.read_csv(path_s2bms)
df_s2bms.aux_corine_frac_lowlevel_top_1

# TMP, captions

In [ ]:
import src.data_preprocessing.caption_generation_s2bms as cg_s2bms
import src.data_preprocessing.caption_generation_satbird as cg_satbird

In [ ]:
cg_s2bms.generate_captions(20, template_type="parallel")

In [ ]:
cg_satbird.generate_captions(20, template_type="parallel")